# 13.2. 군의 표현과 기약표현

In [1]:
import numpy as np


def C(n, axis="z"):
    """각 축에 대해 360/n도만큼 회전"""
    th = 2 * np.pi / n
    c, s = np.cos(th), np.sin(th)
    if axis == "z":
        return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])
    if axis == "y":
        return np.array([[c, 0, s], [0, 1.0, 0], [-s, 0, c]])
    return np.array([[1.0, 0, 0], [0, c, -s], [0, s, c]])


def sigma_v(d):
    """z축을 포함하고 xy-평면과 각도 d를 이루는 평면에 대한 반사"""
    th = np.radians(d)
    return np.array([[np.cos(2 * th), np.sin(2 * th), 0], [np.sin(2 * th), -np.cos(2 * th), 0], [0, 0, 1.0]])


E = np.eye(3)

ops = {
    "E": E,
    "C3": C(3),
    "C3^2": C(3) @ C(3),
    "σ_a": sigma_v(0),
    "σ_b": sigma_v(120),
    "σ_c": sigma_v(240),
}


def which(M, table):
    for name, op in table.items():
        if np.allclose(M, op, atol=1e-9):
            return name
    return "??"


w = 7
print("대칭 조작의 군 곱셈표:")
print(" " * w + "".join(f"{k:>{w}}" for k in ops))
for a, A in ops.items():
    print(f"{a:>{w}}" + "".join(f"{which(A @ B, ops):>{w}}" for B in ops.values()))

대칭 조작의 군 곱셈표:
             E     C3   C3^2    σ_a    σ_b    σ_c
      E      E     C3   C3^2    σ_a    σ_b    σ_c
     C3     C3   C3^2      E    σ_c    σ_a    σ_b
   C3^2   C3^2      E     C3    σ_b    σ_c    σ_a
    σ_a    σ_a    σ_b    σ_c      E     C3   C3^2
    σ_b    σ_b    σ_c    σ_a   C3^2      E     C3
    σ_c    σ_c    σ_a    σ_b     C3   C3^2      E


In [2]:
r, z = 0.9375, -0.3810
ammonia = np.array(
    [
        [0, 0, 0],
        [r, 0, z],
        [r * np.cos(2 * np.pi / 3), r * np.sin(2 * np.pi / 3), z],
        [r * np.cos(4 * np.pi / 3), r * np.sin(4 * np.pi / 3), z],
    ]
)
labels = ["N", "H", "H", "H"]
basis = ["s_N", "s_A", "s_B", "s_C"]


def orbital_rep(op, coords, labels):
    """s 오비탈 기저에 대한 표현 행렬"""
    n = len(coords)
    M = np.zeros((n, n))
    moved = coords @ op.T
    for i, (p, l) in enumerate(zip(moved, labels)):
        for j, (q, m) in enumerate(zip(coords, labels)):
            if l == m and np.allclose(p, q, atol=1e-6):
                M[j, i] = 1
                break
    return M


orb_reps = {k: orbital_rep(v, ammonia, labels) for k, v in ops.items()}
print("표현 행렬의 군 곱셈표:")
print(" " * w + "".join(f"{k:>{w}}" for k in orb_reps))
for a, A in orb_reps.items():
    print(f"{a:>{w}}" + "".join(f"{which(A @ B, orb_reps):>{w}}" for B in orb_reps.values()))

표현 행렬의 군 곱셈표:
             E     C3   C3^2    σ_a    σ_b    σ_c
      E      E     C3   C3^2    σ_a    σ_b    σ_c
     C3     C3   C3^2      E    σ_c    σ_a    σ_b
   C3^2   C3^2      E     C3    σ_b    σ_c    σ_a
    σ_a    σ_a    σ_b    σ_c      E     C3   C3^2
    σ_b    σ_b    σ_c    σ_a   C3^2      E     C3
    σ_c    σ_c    σ_a    σ_b     C3   C3^2      E


In [3]:
# 규격화된 새로운 기저
new_basis = [
    ("sN", np.array([1, 0, 0, 0])),
    ("sA + sB + sC", np.array([0, 1 / np.sqrt(3), 1 / np.sqrt(3), 1 / np.sqrt(3)])),
    ("2sA - sB - sC", np.array([0, 2 / np.sqrt(6), -1 / np.sqrt(6), -1 / np.sqrt(6)])),
    ("sB - sC", np.array([0, 0, 1 / np.sqrt(2), -1 / np.sqrt(2)])),
]

new_basis_names = [b[0] for b in new_basis]
T = np.column_stack([b[1] for b in new_basis])

print("T는 직교 행렬인가:", np.allclose(T.T @ T, np.eye(4)))
print()
w = 16
for r in ["C3", "σ_a"]:
    print(f"새 기저에서 {r}의 표현 행렬")
    print(" " * w + "".join(f"{k:>{w}}" for k in new_basis_names))
    for b, row in zip(new_basis_names, T.T @ orb_reps[r] @ T):
        clean = [0.0 if abs(v) < 1e-12 else v for v in row]
        print(f"{b:>{w}}" + "".join(f"{v:>16.3f}" for v in clean))
    print()

T는 직교 행렬인가: True

새 기저에서 C3의 표현 행렬
                              sN    sA + sB + sC   2sA - sB - sC         sB - sC
              sN           1.000           0.000           0.000           0.000
    sA + sB + sC           0.000           1.000           0.000           0.000
   2sA - sB - sC           0.000           0.000          -0.500          -0.866
         sB - sC           0.000           0.000           0.866          -0.500

새 기저에서 σ_a의 표현 행렬
                              sN    sA + sB + sC   2sA - sB - sC         sB - sC
              sN           1.000           0.000           0.000           0.000
    sA + sB + sC           0.000           1.000           0.000           0.000
   2sA - sB - sC           0.000           0.000           1.000           0.000
         sB - sC           0.000           0.000           0.000          -1.000



In [4]:
C3_rep = (T.T @ orb_reps["C3"] @ T)[2:, 2:]
sigma_a_rep = (T.T @ orb_reps["σ_a"] @ T)[2:, 2:]

print("회전 후 반사:")
print(np.round(C3_rep @ sigma_a_rep, 3))
print()
print("반사 후 회전:")
print(np.round(sigma_a_rep @ C3_rep, 3))

회전 후 반사:
[[-0.5    0.866]
 [ 0.866  0.5  ]]

반사 후 회전:
[[-0.5   -0.866]
 [-0.866  0.5  ]]
